# Khmer Legal Chunk Search

Retrieve small context chunks from ChromaDB and keep the parent article identifiers for question answering.

In [ ]:
import gc
import json
import logging
import unicodedata
from pathlib import Path

logging.getLogger(
    'torch.distributed.elastic.multiprocessing.redirects'
).setLevel(logging.ERROR)

import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer


def find_project_root(start: Path) -> Path:
    candidates = (start, *start.parents)
    for directory in candidates:
        if (directory / 'data-pipeline').is_dir():
            return directory
    raise RuntimeError('Could not find the project root')


PROJECT_ROOT = find_project_root(Path.cwd())
ARTICLE_PATH = (
    PROJECT_ROOT
    / 'data-pipeline'
    / 'data'
    / 'processed'
    / 'search'
    / 'articles_normalized.jsonl'
)
CHROMA_DIR = (
    PROJECT_ROOT
    / 'data-pipeline'
    / 'data'
    / 'processed'
    / 'chroma_db'
)
ARTICLES_BY_ID = {
    record['id']: record
    for line in ARTICLE_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
    for record in [json.loads(line)]
}



In [ ]:
def encode_huggingface_encoder(query: str, model_name: str) -> np.ndarray:
    import torch
    from transformers import AutoModel, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded = tokenizer([query], padding=True, truncation=True, max_length=512, return_tensors='pt')
    with torch.no_grad():
        output = model(**encoded).last_hidden_state
    mask = encoded['attention_mask'].unsqueeze(-1).float()
    vector = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    vector = vector / vector.norm(dim=1, keepdim=True)
    del model, tokenizer
    gc.collect()
    return vector[0].cpu().numpy().astype('float32')

MODELS = {
    'sealion': {
        'collection': 'legal_chunks_sealion',
        'model': 'aisingapore/SEA-LION-E5-Embedding-600M',
    },
    'granite-r2': {
        'collection': 'legal_chunks_granite_r2',
        'model': 'ibm-granite/granite-embedding-311m-multilingual-r2',
    },
}
MODEL_KEYS = ['sealion', 'granite-r2']
TOP_K = 5


def normalize_query(query: str) -> str:
    query = unicodedata.normalize('NFC', query)
    query = query.replace('\u200b', '')
    query = query.replace('\u200c', '')
    query = query.replace('\u200d', '')
    query = query.replace('\u00a0', ' ')
    return ' '.join(query.split())


def embed_query(query: str, model_key: str) -> np.ndarray:
    model_name = MODELS[model_key]['model']
    if model_key == 'albert':
        vector = encode_huggingface_encoder(query, model_name)
    else:
        model = SentenceTransformer(model_name)
        vector = model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )[0]
    return np.asarray(vector, dtype='float32')


def search_articles_from_chunks(
    query: str,
    model_key: str,
    top_k: int = TOP_K,
) -> list[dict]:
    query = normalize_query(query)
    if not query:
        raise ValueError('Query cannot be empty')

    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    collection = client.get_collection(MODELS[model_key]['collection'])
    query_vector = embed_query(query, model_key)
    result = collection.query(
        query_embeddings=[query_vector.tolist()],
        n_results=max(top_k * 3, top_k),
        include=['documents', 'metadatas', 'distances'],
    )

    matches = []
    seen_parent_ids = set()
    for rank, chunk_id in enumerate(result['ids'][0]):
        metadata = result['metadatas'][0][rank] or {}
        parent_id = metadata.get('parent_id') or chunk_id.rsplit('-chunk-', 1)[0]
        if parent_id in seen_parent_ids:
            continue
        article = ARTICLES_BY_ID.get(parent_id)
        if article is None:
            article = next((record for record in ARTICLES_BY_ID.values() if record.get('article_id') == metadata.get('article_id') and record.get('document_id') == metadata.get('document_id')), None)
        if article is None:
            continue
        seen_parent_ids.add(parent_id)
        matches.append({
            'rank': len(matches) + 1,
            'parent_id': parent_id,
            'chunk_id': chunk_id,
            'article_id': article.get('article_id'),
            'document_id': article.get('document_id'),
            'article_number': article.get('article_number'),
            'distance': result['distances'][0][rank],
            'matched_chunk_text': result['documents'][0][rank],
            'text': article.get('text', ''),
            'title': article.get('document_title'),
            'source_url': article.get('source_url'),
            'pdf_url': article.get('pdf_url'),
        })
    return matches[:top_k]

In [ ]:
query = "បើខ្ញុំបើកបររថយន្តពេលស្រវឹង តើអាចត្រូវពិន័យ ឬទទួលទោសអ្វីខ្លះ?"

all_results = {model_key: search_articles_from_chunks(query, model_key) for model_key in MODEL_KEYS}

print(json.dumps(all_results, ensure_ascii=False, indent=2))